In [1]:
import torch 
import torch.nn as nn 
import math 

In [2]:
d_model = 8
num_heads = 2

d_k = d_model // num_heads

In [3]:
X = torch.tensor([
    [1., 0., 1., 0., 1., 0., 1., 0.],
    [0., 1., 0., 1., 0., 1., 0., 1.],
    [1., 1., 0., 0., 1., 1., 0., 0.]
])

In [4]:
W_Q = nn.Parameter(torch.randn(d_model, d_model))
W_K = nn.Parameter(torch.randn(d_model, d_model))
W_V = nn.Parameter(torch.randn(d_model, d_model))

In [5]:
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

In [6]:
Q = Q.view(3, num_heads, d_k)
K = K.view(3, num_heads, d_k)
V = V.view(3, num_heads, d_k)

In [7]:
Q = Q.transpose(0, 1)
K = K.transpose(0, 1)
V = V.transpose(0, 1)

In [8]:
scores = Q @ K.transpose(-2, -1)

In [9]:
scores = scores / math.sqrt(d_k)

attention_weights = torch.softmax(scores, dim=-1)

head_outputs = attention_weights @ V

In [10]:
head_outputs = head_outputs.transpose(0, 1)

In [11]:
concat = head_outputs.contiguous().view(3, d_model)

In [12]:
W_O = nn.Parameter(torch.randn(d_model, d_model))

output = concat @ W_O

In [13]:
output

tensor([[-0.8320, -4.9815,  4.4039,  0.8585, -1.7073, -3.9190,  0.5433,  0.5939],
        [-2.2584,  0.2033, -2.4484, -1.7649, -4.8205,  0.3776,  9.1498, -3.0623],
        [-1.6934, -0.2820, -2.7247, -1.0765, -5.1214, -1.3151,  7.5756, -2.1535]],
       grad_fn=<MmBackward0>)

In [14]:
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        # Overall representation size
        # Example: d_model = 8
        self.d_model = d_model

        # Number of attention heads
        # Example: num_heads = 2
        self.num_heads = num_heads

        # d_model must be divisible by num_heads
        #
        # Example:
        # d_model = 8
        # num_heads = 2
        #
        # 8 / 2 = 4 features per head
        assert d_model % num_heads == 0

        # Dimension of Q and K inside each head
        #
        # Example:
        # d_k = 8 / 2 = 4
        self.d_k = d_model // num_heads

        # --------------------------------------------------
        # Learnable projections
        # --------------------------------------------------

        # X → Q
        #
        # Input:
        # (B, T, d_model)
        #
        # Output:
        # (B, T, d_model)
        self.W_Q = nn.Linear(d_model, d_model)

        # X → K
        #
        # (B, T, d_model)
        #       ↓
        # (B, T, d_model)
        self.W_K = nn.Linear(d_model, d_model)

        # X → V
        #
        # (B, T, d_model)
        #       ↓
        # (B, T, d_model)
        self.W_V = nn.Linear(d_model, d_model)

        # Final output projection
        #
        # Concatenated heads:
        # (B, T, d_model)
        #
        #       ↓ W_O
        #
        # Output:
        # (B, T, d_model)
        self.W_O = nn.Linear(d_model, d_model)


    def forward(self, X):

        # --------------------------------------------------
        # INPUT
        # --------------------------------------------------
        #
        # X shape:
        #
        # (B, T, d_model)
        #
        # Example:
        # (2, 5, 8)
        #
        # 2 = batch size
        # 5 = number of tokens
        # 8 = features per token

        B, T, _ = X.shape


        # --------------------------------------------------
        # STEP 1: Create Q, K, V
        # --------------------------------------------------

        Q = self.W_Q(X)

        # X:
        # (B, T, d_model)
        #      ↓
        # Q:
        # (B, T, d_model)
        #
        # Example:
        # (2, 5, 8)

        K = self.W_K(X)

        # K:
        # (B, T, d_model)
        #
        # Example:
        # (2, 5, 8)

        V = self.W_V(X)

        # V:
        # (B, T, d_model)
        #
        # Example:
        # (2, 5, 8)


        # --------------------------------------------------
        # STEP 2: Split Q, K, V into multiple heads
        # --------------------------------------------------

        Q = Q.view(
            B,
            T,
            self.num_heads,
            self.d_k
        )

        # Before:
        # Q = (B, T, d_model)
        #   = (2, 5, 8)
        #
        # After:
        # Q = (B, T, num_heads, d_k)
        #   = (2, 5, 2, 4)
        #
        # Because:
        #
        # 8 features
        #      ↓
        # 2 heads × 4 features

        K = K.view(
            B,
            T,
            self.num_heads,
            self.d_k
        )

        # Before:
        # K = (2, 5, 8)
        #
        # After:
        # K = (2, 5, 2, 4)

        V = V.view(
            B,
            T,
            self.num_heads,
            self.d_k
        )

        # Before:
        # V = (2, 5, 8)
        #
        # After:
        # V = (2, 5, 2, 4)


        # --------------------------------------------------
        # STEP 3: Move heads before sequence dimension
        # --------------------------------------------------

        Q = Q.transpose(1, 2)

        # Before:
        # Q = (B, T, heads, d_k)
        #   = (2, 5, 2, 4)
        #
        # transpose(1, 2)
        #
        # swaps:
        #     T ↔ heads
        #
        # After:
        # Q = (B, heads, T, d_k)
        #   = (2, 2, 5, 4)

        K = K.transpose(1, 2)

        # Before:
        # K = (2, 5, 2, 4)
        #
        # After:
        # K = (2, 2, 5, 4)

        V = V.transpose(1, 2)

        # Before:
        # V = (2, 5, 2, 4)
        #
        # After:
        # V = (2, 2, 5, 4)


        # --------------------------------------------------
        # STEP 4: Calculate attention scores
        # --------------------------------------------------

        scores = Q @ K.transpose(-2, -1)

        # Q:
        # (B, heads, T, d_k)
        # (2, 2, 5, 4)
        #
        # K:
        # (B, heads, T, d_k)
        # (2, 2, 5, 4)
        #
        # K.transpose(-2, -1):
        # (B, heads, d_k, T)
        # (2, 2, 4, 5)
        #
        # Matrix multiplication:
        #
        # (2, 2, 5, 4)
        #             ×
        # (2, 2, 4, 5)
        #
        #             ↓
        #
        # scores:
        # (2, 2, 5, 5)
        #
        # Meaning:
        # B       = 2 batches
        # heads   = 2 attention heads
        # 5 × 5   = token-to-token attention matrix


        # --------------------------------------------------
        # STEP 5: Scale the scores
        # --------------------------------------------------

        scores = scores / math.sqrt(self.d_k)

        # Before:
        # scores = (B, heads, T, T)
        #        = (2, 2, 5, 5)
        #
        # d_k = 4
        #
        # divide by:
        # sqrt(4) = 2
        #
        # After:
        # scores = (2, 2, 5, 5)
        #
        # Shape does NOT change.
        # Only the numerical values change.


        # --------------------------------------------------
        # STEP 6: Softmax
        # --------------------------------------------------

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Before:
        # scores:
        # (B, heads, T, T)
        # (2, 2, 5, 5)
        #
        # Softmax is applied across the LAST dimension.
        #
        # After:
        # attention_weights:
        # (2, 2, 5, 5)
        #
        # Each row contains attention probabilities
        # whose values sum to 1.


        # --------------------------------------------------
        # STEP 7: Weighted sum of V
        # --------------------------------------------------

        head_outputs = attention_weights @ V

        # attention_weights:
        # (B, heads, T, T)
        # (2, 2, 5, 5)
        #
        # V:
        # (B, heads, T, d_k)
        # (2, 2, 5, 4)
        #
        # Matrix multiplication:
        #
        # (2, 2, 5, 5)
        #             ×
        # (2, 2, 5, 4)
        #
        #             ↓
        #
        # head_outputs:
        # (B, heads, T, d_k)
        # (2, 2, 5, 4)


        # --------------------------------------------------
        # STEP 8: Combine all heads
        # --------------------------------------------------

        head_outputs = head_outputs.transpose(1, 2)

        # Before:
        # (B, heads, T, d_k)
        # (2, 2, 5, 4)
        #
        # After:
        # (B, T, heads, d_k)
        # (2, 5, 2, 4)

        head_outputs = head_outputs.contiguous()

        # Same shape:
        #
        # (2, 5, 2, 4)
        #
        # contiguous() does NOT change the shape.
        # It makes the tensor memory layout suitable
        # for operations such as view().


        concat = head_outputs.view(
            B,
            T,
            self.d_model
        )

        # Before:
        # (B, T, heads, d_k)
        # (2, 5, 2, 4)
        #
        # Combine:
        #
        # heads × d_k
        #
        # 2 × 4 = 8
        #
        # After:
        # concat:
        # (B, T, d_model)
        # (2, 5, 8)


        # --------------------------------------------------
        # STEP 9: Final output projection
        # --------------------------------------------------

        output = self.W_O(concat)

        # Before:
        # concat:
        # (B, T, d_model)
        # (2, 5, 8)
        #
        # W_O:
        # 8 → 8
        #
        # After:
        # output:
        # (B, T, d_model)
        # (2, 5, 8)


        return output

In [18]:
X = torch.randn(2, 5, 8)

mha = MultiHeadAttention(
    d_model=8,
    num_heads=2
)

output = mha(X)

print(X.shape)
print(output.shape)

torch.Size([2, 5, 8])
torch.Size([2, 5, 8])
